# Model-Agnostic ASR Fine-Tuning Environment
**Change `MODEL_NAME` in Cell 2 and everything else stays the same.**

Registered models:
- `omnilingual-asr/omniASR_LLM_1B`
- `omnilingual-asr/omniASR_LLM_300M`

(Whisper / Qwen3-ASR / CTC adapters are stubbed in the registry so the same APIs extend to them.)

### Architecture
```
ConfigAPI  ->  per-model LoRA + TrainingArguments defaults
ModelAdapter (ABC)
   |-- preprocess(batch)      # model-specific: chat template / CTC labels / raw features
   |-- collate(features)      # model-specific data collator
   |-- load_base()            # local-first, then hub, then cached locally
   |-- generate(batch)        # decoding
   |-- apply_lora(cfg)        # Unsloth where supported, PEFT otherwise
PredictAPI   -> cached preds  (skip if exists)
EvaluateAPI  -> cached WER/CER (skip if exists)
TrainAPI     -> 50 epochs, early stop patience=4, best-WER checkpoint, W&B
```

### Two things you should know up front
1. **OmniASR LLM is fairseq2, not `transformers`.** There is no `AutoModelForSpeechSeq2Seq` path. We load through `omnilingual_asr` and wrap the underlying `torch.nn.Module`.
2. **Unsloth does not support wav2vec2_llama.** `apply_lora` tries Unsloth first and falls back to plain PEFT for OmniASR. This is not a workaround to be removed later — Unsloth has no kernel path for this architecture.

In [ ]:
# Cell 1 — Environment
!pip -q install "omnilingual-asr" peft transformers datasets jiwer wandb accelerate soundfile librosa evaluate
# Unsloth only used for architectures it supports (Whisper/Qwen). Safe to skip on OmniASR-only runs.
# !pip -q install unsloth

import os, json, gc, math, time, random, hashlib, warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List, Optional, Callable

import numpy as np, torch, torch.nn as nn
warnings.filterwarnings("ignore")
print(torch.__version__, torch.cuda.is_available())

## Cell 2 — The only line you change

In [ ]:
MODEL_NAME = "omnilingual-asr/omniASR_LLM_1B"
# MODEL_NAME = "omnilingual-asr/omniASR_LLM_300M"

LANG          = "arb_Arab"          # OmniASR language token (Arabic, Arabic script)
SMOKE_TEST    = True                # tiny subsets + 2 epochs
SEED          = 42

# Device-agnostic: real runs are on GPU, but the harness must also import/run on CPU boxes.
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

# ROOT defaults to the RunPod network volume but is overridable for local/off-RunPod runs.
ROOT          = Path(os.environ.get("ASR_ENV_ROOT", "/workspace/asr_env"))
MODEL_CACHE   = ROOT / "models"
PRED_DIR      = ROOT / "preds"
METRIC_DIR    = ROOT / "metrics"
CKPT_DIR      = ROOT / "checkpoints"
for d in (MODEL_CACHE, PRED_DIR, METRIC_DIR, CKPT_DIR): d.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(MODEL_CACHE / "hf")
os.environ["WANDB_PROJECT"] = "arabic-asr-modelagnostic"

def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
print(f"DEVICE={DEVICE} | ROOT={ROOT}")

## Cell 3 — Arabic normalization + WER/CER

In [ ]:
import re, unicodedata, jiwer

_DIAC = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0640]")
_PUNC = re.compile(r"[^\w\s\u0621-\u064A]")

def normalize_ar(t: str) -> str:
    """Diacritic strip, tatweel removal, alef/ya/ta-marbuta unification."""
    if t is None: return ""
    t = unicodedata.normalize("NFKC", str(t))
    t = _DIAC.sub("", t)
    t = re.sub("[\u0622\u0623\u0625\u0671]", "\u0627", t)   # alef variants -> alef
    t = t.replace("\u0649", "\u064A")                          # alef maqsura -> ya
    t = t.replace("\u0629", "\u0647")                          # ta marbuta -> ha
    t = t.replace("\u0624", "\u0648").replace("\u0626", "\u064A")
    t = _PUNC.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()

def compute_wer_cer(preds, refs, normalize=True):
    if normalize:
        preds = [normalize_ar(p) for p in preds]
        refs  = [normalize_ar(r) for r in refs]
    keep = [(p, r) for p, r in zip(preds, refs) if r.strip()]
    if not keep: return {"wer": float("nan"), "cer": float("nan"), "n": 0}
    p, r = zip(*keep)
    return {"wer": jiwer.wer(list(r), list(p)),
            "cer": jiwer.cer(list(r), list(p)),
            "n": len(r)}

## Cell 4 — ConfigAPI

Per-model LoRA + training defaults. Your Whisper study settled on `r=32, alpha=32, lr=1e-4, AdamW 8-bit, patience=3` — here patience is 4 per spec and the OmniASR target modules follow the wav2vec2_llama attention naming.

In [ ]:
@dataclass
class LoRAConfigSpec:
    r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    bias: str = "none"
    target_modules: List[str] = field(default_factory=lambda: ["q_proj","k_proj","v_proj","output_proj"])
    modules_to_save: Optional[List[str]] = None
    task_type: Optional[str] = None

@dataclass
class TrainConfigSpec:
    num_epochs: int = 50
    early_stopping_patience: int = 4
    metric_for_best: str = "wer"
    greater_is_better: bool = False
    per_device_train_batch_size: int = 4
    per_device_eval_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float = 1e-4
    warmup_ratio: float = 0.05
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    optim: str = "adamw_bnb_8bit"
    bf16: bool = True
    gradient_checkpointing: bool = True
    dataloader_num_workers: int = 4
    max_audio_seconds: float = 30.0
    max_label_tokens: int = 256
    save_total_limit: int = 2

class ConfigAPI:
    """Single source of truth for per-model hyperparameters.

    NOTE on target_modules: OmniASR LLM is fairseq2, not transformers. Attention =
    StandardMultiheadAttention (q_proj/k_proj/v_proj/output_proj); FFN = GLUFeedForwardNetwork
    (gate_proj/inner_proj/output_proj). The old transformers-style names
    (o_proj/up_proj/down_proj) do NOT exist here. `OmniASRAdapter` also re-derives these from
    the live `named_modules()` at load time and warns if the config names miss.
    """
    _LORA = {
        "omnilingual-asr/omniASR_LLM_1B": LoRAConfigSpec(
            r=32, lora_alpha=32,
            target_modules=["q_proj","k_proj","v_proj","output_proj","gate_proj","inner_proj"]),
        "omnilingual-asr/omniASR_LLM_300M": LoRAConfigSpec(
            r=32, lora_alpha=32,
            target_modules=["q_proj","k_proj","v_proj","output_proj","gate_proj","inner_proj"]),
    }
    _TRAIN = {
        "omnilingual-asr/omniASR_LLM_1B":   TrainConfigSpec(per_device_train_batch_size=2,
                                                            gradient_accumulation_steps=8),
        "omnilingual-asr/omniASR_LLM_300M": TrainConfigSpec(per_device_train_batch_size=4,
                                                            gradient_accumulation_steps=4),
    }
    @classmethod
    def lora(cls, name)  -> LoRAConfigSpec:  return cls._LORA.get(name, LoRAConfigSpec())
    @classmethod
    def train(cls, name) -> TrainConfigSpec: return cls._TRAIN.get(name, TrainConfigSpec())

print(json.dumps(asdict(ConfigAPI.lora(MODEL_NAME)), indent=2))
print(json.dumps(asdict(ConfigAPI.train(MODEL_NAME)), indent=2))

## Cell 5 — ModelAdapter ABC

The whole point of the abstraction. Every model-specific difference lives behind one of these six methods:

| method | Whisper | Qwen3-ASR | CTC (NemoCTC) | OmniASR LLM |
|---|---|---|---|---|
| `preprocess` | log-mel + tokenized text | **chat template** wrap | text -> char ids, no BOS/EOS | audio tensor + `lang` token prefix |
| `collate` | pad mel + `-100` label pad | pad ids + attn mask | pad audio + label lens | pad waveform + pad label ids |
| `loss_type` | seq2seq CE | seq2seq CE | **CTC** | seq2seq CE |

In [ ]:
from abc import ABC, abstractmethod

class ModelAdapter(ABC):
    name: str
    loss_type: str = "seq2seq"          # or "ctc"
    supports_unsloth: bool = False

    def __init__(self, model_name: str, lang: str = LANG):
        self.model_name = model_name; self.lang = lang
        self.model = None; self.processor = None

    @abstractmethod
    def load_base(self): ...
    @abstractmethod
    def preprocess(self, example: Dict) -> Dict: ...
    @abstractmethod
    def collate(self, features: List[Dict]) -> Dict[str, torch.Tensor]: ...
    @abstractmethod
    def generate(self, batch: Dict) -> List[str]: ...

    def apply_lora(self, spec: LoRAConfigSpec):
        """Unsloth-first, PEFT fallback. OmniASR has no Unsloth path — that's expected."""
        if self.supports_unsloth:
            try:
                from unsloth import FastModel
                self.model = FastModel.get_peft_model(
                    self.model, r=spec.r, lora_alpha=spec.lora_alpha,
                    lora_dropout=spec.lora_dropout, bias=spec.bias,
                    target_modules=spec.target_modules, use_gradient_checkpointing="unsloth")
                print("[lora] unsloth"); return self.model
            except Exception as e:
                print(f"[lora] unsloth unavailable ({e}); falling back to PEFT")
        from peft import LoraConfig, get_peft_model
        kw = dict(r=spec.r, lora_alpha=spec.lora_alpha, lora_dropout=spec.lora_dropout,
                  bias=spec.bias, target_modules=spec.target_modules)
        if spec.modules_to_save: kw["modules_to_save"] = spec.modules_to_save
        if spec.task_type:       kw["task_type"] = spec.task_type
        self.model = get_peft_model(self.model, LoraConfig(**kw))
        self.model.print_trainable_parameters()
        print("[lora] peft"); return self.model

    def train_step(self, batch) -> torch.Tensor:
        out = self.model(**batch)
        return out.loss if hasattr(out, "loss") else out["loss"]

## Cell 6 — OmniASR adapter (shared by 1B and 300M)

One class, two entries in the registry. The only delta is the `model_card` string, exactly as you predicted.

In [ ]:
import math

class _LoRALinear(nn.Module):
    """Manual LoRA wrapper. OmniASR's projections are `fairseq2.nn.projection.Linear`, which is
    NOT a `torch.nn.Linear` subclass, so neither PEFT nor Unsloth can wrap them. We inject a
    low-rank side path ourselves: y = base(x) + scaling * dropout(x) @ A^T @ B^T, B zero-init so
    the initial delta is 0. Works on any module exposing a 2-D `.weight`."""
    def __init__(self, base: nn.Module, r: int, alpha: int, dropout: float):
        super().__init__()
        self.base = base
        for p in self.base.parameters(): p.requires_grad_(False)
        out_f, in_f = base.weight.shape
        dt, dev = base.weight.dtype, base.weight.device
        self.lora_A = nn.Parameter(torch.zeros(r, in_f, dtype=dt, device=dev))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r, dtype=dt, device=dev))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.scaling = alpha / r
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        out = self.base(x)
        delta = self.drop(x) @ self.lora_A.t() @ self.lora_B.t()
        return out + self.scaling * delta


class OmniASRAdapter(ModelAdapter):
    """fairseq2 wav2vec2_llama. Verified against omnilingual-asr@main source AND a real CPU run
    of omniASR_LLM_300M (1.63B params). See DISCOVERY.md / SMOKE_RESULTS.md.

      * pipeline.model -> Wav2Vec2LlamaModel;  pipeline.tokenizer -> Tokenizer
      * tokenizer.create_encoder() takes NO lang; decode via create_decoder(skip_special_tokens=True)
      * TRAINING: loss = model(Seq2SeqBatch(...)); model builds `audio [lang] <bos> text <eos>` and
        masks the loss internally. Pad text with pad_idx (NOT -100). seq_lens must be list[int].
      * INFERENCE: pipeline.transcribe(list[dict{waveform,sample_rate}], lang=[...], batch_size=n)
      * LoRA: projections are fairseq2.nn.projection.Linear (NOT torch.nn.Linear) -> PEFT/Unsloth
        can't wrap them, so we inject LoRA manually via _LoRALinear.
    """
    loss_type = "seq2seq"
    supports_unsloth = False

    CARD = {"omnilingual-asr/omniASR_LLM_1B":   "omniASR_LLM_1B",
            "omnilingual-asr/omniASR_LLM_300M": "omniASR_LLM_300M"}

    def __init__(self, model_name, lang=LANG):
        super().__init__(model_name, lang)
        self.card = self.CARD[model_name]
        self.pipeline = None; self.tokenizer = None
        self.pad_idx = 0
        self._encoder = None; self._decoder = None
        self.derived_target_modules = None

    def load_base(self):
        local = MODEL_CACHE / self.card
        from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
        if local.exists() and any(local.iterdir()):
            print(f"[load] local -> {local}")
        else:
            print(f"[load] downloading {self.card} -> {local}")
            local.mkdir(parents=True, exist_ok=True)
        # FAIRSEQ2_CACHE_DIR controls the checkpoint cache (verified on a real 300M load).
        os.environ.setdefault("FAIRSEQ2_CACHE_DIR", str(local))

        self.pipeline  = ASRInferencePipeline(self.card, device=DEVICE)
        self.model     = self.pipeline.model
        self.tokenizer = self.pipeline.tokenizer
        self._encoder  = self.tokenizer.create_encoder()
        self._decoder  = self.tokenizer.create_decoder(skip_special_tokens=True)
        self.pad_idx   = getattr(self.tokenizer.vocab_info, "pad_idx", 0) or 0

        from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs
        assert self.lang in supported_langs, f"{self.lang} not in supported_langs"

        self.derived_target_modules = self._derive_target_modules()
        print(f"[load] pad_idx={self.pad_idx} | derived target_modules={self.derived_target_modules}")
        return self.model

    def _derive_target_modules(self):
        """Detect projection leaf names by DUCK TYPING (2-D `.weight`), because fairseq2's Linear
        is not a torch.nn.Linear subclass and isinstance(mod, nn.Linear) would miss all of them."""
        want = {"q_proj","k_proj","v_proj","output_proj","gate_proj","inner_proj"}
        found = set()
        for name, mod in self.model.named_modules():
            leaf = name.split(".")[-1]
            w = getattr(mod, "weight", None)
            if leaf in want and w is not None and getattr(w, "ndim", 0) == 2:
                found.add(leaf)
        return sorted(found) or sorted(want)

    def apply_lora(self, spec: LoRAConfigSpec):
        """Manual LoRA injection (PEFT/Unsloth can't wrap fairseq2.nn.projection.Linear)."""
        targets = set(self.derived_target_modules or spec.target_modules)
        replaced = 0
        for mod_name, mod in list(self.model.named_modules()):
            leaf = mod_name.split(".")[-1]
            w = getattr(mod, "weight", None)
            if (leaf in targets and w is not None and getattr(w, "ndim", 0) == 2
                    and not isinstance(mod, _LoRALinear)):
                parent = self.model.get_submodule(mod_name.rsplit(".", 1)[0]) if "." in mod_name else self.model
                setattr(parent, leaf, _LoRALinear(mod, spec.r, spec.lora_alpha, spec.lora_dropout))
                replaced += 1
        for n, p in self.model.named_parameters():
            p.requires_grad_("lora_" in n)
        n_tr = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"[lora] manual injection into {replaced} fairseq2 Linear layers | trainable params={n_tr}")
        return self.model

    def preprocess(self, ex):
        audio = ex["audio"]["array"]; sr = ex["audio"]["sampling_rate"]
        if sr != 16000:
            import librosa; audio = librosa.resample(np.asarray(audio, dtype=np.float32),
                                                     orig_sr=sr, target_sr=16000)
        text = normalize_ar(ex["text"])
        ids  = self._encode(text)
        return {"input_values": np.asarray(audio, dtype=np.float32),
                "labels": ids, "text": text,
                "audio_len": len(audio) / 16000.0}

    def _encode(self, text):
        if self._encoder is None: return []
        return self._encoder(text).tolist()

    def _decode(self, ids):
        if self._decoder is None: return ""
        import torch as _t
        return str(self._decoder(_t.as_tensor(ids, dtype=_t.int64)))

    def collate(self, feats):
        maxa = max(len(f["input_values"]) for f in feats)
        maxl = max(len(f["labels"]) for f in feats) or 1
        wav  = torch.zeros(len(feats), maxa, dtype=torch.float32)
        mask = torch.zeros(len(feats), maxa, dtype=torch.long)
        lab  = torch.full((len(feats), maxl), self.pad_idx, dtype=torch.long)
        lab_lens = torch.zeros(len(feats), dtype=torch.long)
        for i, f in enumerate(feats):
            a = torch.as_tensor(f["input_values"], dtype=torch.float32)
            wav[i, :len(a)] = a; mask[i, :len(a)] = 1
            if len(f["labels"]):
                lab[i, :len(f["labels"])] = torch.as_tensor(f["labels"], dtype=torch.long)
                lab_lens[i] = len(f["labels"])
        return {"input_values": wav, "attention_mask": mask, "labels": lab,
                "label_lengths": lab_lens,
                "lang": [self.lang]*len(feats), "text": [f["text"] for f in feats]}

    def train_step(self, batch) -> torch.Tensor:
        from fairseq2.datasets.batch import Seq2SeqBatch
        wav  = batch["input_values"]; mask = batch["attention_mask"]; lab = batch["labels"]
        # fairseq2 Seq2SeqBatch requires seq_lens as list[int], NOT tensors.
        src_lens = mask.sum(dim=1).to(torch.long).tolist()
        if "label_lengths" in batch:
            tgt_lens = batch["label_lengths"].to(torch.long).tolist()
        else:
            tgt_lens = (lab != self.pad_idx).sum(dim=1).to(torch.long).tolist()
        model_dtype = next(self.model.parameters()).dtype
        dev = getattr(self.model, "device", DEVICE)
        langs = batch.get("lang", [self.lang]*wav.shape[0])
        seq2seq = Seq2SeqBatch(
            source_seqs     = wav.to(dev, model_dtype),
            source_seq_lens = src_lens,
            target_seqs     = lab.to(dev).to(torch.long),
            target_seq_lens = tgt_lens,
            example         = {"lang": list(langs)},
        )
        out = self.model(seq2seq)
        return out if torch.is_tensor(out) else out[0]

    @torch.no_grad()
    def generate(self, batch):
        wav  = batch["input_values"]; mask = batch["attention_mask"]
        inp = []
        for i in range(wav.shape[0]):
            n = int(mask[i].sum().item())
            inp.append({"waveform": wav[i, :n].float().cpu().numpy(), "sample_rate": 16000})
        langs = batch.get("lang", [self.lang]*len(inp))
        out = self.pipeline.transcribe(inp, lang=list(langs), batch_size=len(inp))
        return [str(o) for o in out]

## Cell 7 — Registry (extension points for Whisper / Qwen / CTC)

In [ ]:
class WhisperAdapter(ModelAdapter):
    """No chat template. Log-mel features + tokenized labels."""
    supports_unsloth = True
    def load_base(self): raise NotImplementedError("register when needed")
    def preprocess(self, ex): raise NotImplementedError
    def collate(self, f): raise NotImplementedError
    def generate(self, b): raise NotImplementedError

class QwenASRAdapter(ModelAdapter):
    """Chat template REQUIRED — this is the key preprocess divergence."""
    supports_unsloth = True
    def preprocess(self, ex):
        msgs = [{"role":"user","content":[{"type":"audio","audio_url":"<audio>"},
                                          {"type":"text","text":"Transcribe the Arabic audio."}]},
                {"role":"assistant","content":normalize_ar(ex["text"])}]
        # self.processor.apply_chat_template(msgs, tokenize=True, ...)
        raise NotImplementedError("register when needed")
    def load_base(self): raise NotImplementedError
    def collate(self, f): raise NotImplementedError
    def generate(self, b): raise NotImplementedError

class CTCAdapter(ModelAdapter):
    """loss_type='ctc' -> labels are char ids, no BOS/EOS, greedy argmax decode."""
    loss_type = "ctc"; supports_unsloth = False
    def load_base(self): raise NotImplementedError
    def preprocess(self, ex): raise NotImplementedError
    def collate(self, f): raise NotImplementedError
    def generate(self, b): raise NotImplementedError

REGISTRY: Dict[str, Callable[..., ModelAdapter]] = {
    "omnilingual-asr/omniASR_LLM_1B":   OmniASRAdapter,
    "omnilingual-asr/omniASR_LLM_300M": OmniASRAdapter,
    # "openai/whisper-large-v3": WhisperAdapter,
    # "Qwen/Qwen3-ASR":         QwenASRAdapter,
    # "nvidia/stt_ar_fastconformer_ctc": CTCAdapter,
}

def get_adapter(name, **kw) -> ModelAdapter:
    if name not in REGISTRY: raise KeyError(f"{name} not registered. Have: {list(REGISTRY)}")
    a = REGISTRY[name](name, **kw); a.name = name; return a

## Cell 8 — Datasets

Assumed ready. Swap the loader for your QASR/MGB2 splits.

In [ ]:
from datasets import load_dataset, Audio, Dataset

def load_splits(smoke=SMOKE_TEST):
    # >>> replace with your prepared QASR / MGB2 splits <<<
    # ds = load_dataset("parquet", data_files={"train": ..., "validation": ..., "test": ...})
    ds = load_dataset("mozilla-foundation/common_voice_17_0", "ar", trust_remote_code=True)
    ds = ds.rename_column("sentence", "text") if "sentence" in ds["train"].column_names else ds
    splits = {"train": ds["train"], "validation": ds["validation"], "test": ds["test"]}
    if smoke:
        splits = {k: v.shuffle(seed=SEED).select(range(min(n, len(v))))
                  for (k, v), n in zip(splits.items(), [16, 8, 8])}
    for k in splits: splits[k] = splits[k].cast_column("audio", Audio(sampling_rate=16000))
    return splits

SPLITS = load_splits()
{k: len(v) for k, v in SPLITS.items()}

## Cell 9 — PredictAPI (cached)

Cache key = `model + split + dataset fingerprint + stage`. Re-running the cell loads from disk instead of re-decoding.

In [ ]:
def _fingerprint(ds) -> str:
    try: h = ds._fingerprint
    except Exception: h = str(len(ds))
    return hashlib.md5(f"{h}{len(ds)}".encode()).hexdigest()[:10]

class PredictAPI:
    @staticmethod
    def _path(model_name, split, ds, stage):
        slug = model_name.replace("/", "__")
        return PRED_DIR / f"{slug}__{split}__{_fingerprint(ds)}__{stage}.json"

    @staticmethod
    def run(adapter, ds, split="test", stage="base", batch_size=4, force=False):
        p = PredictAPI._path(adapter.name, split, ds, stage)
        if p.exists() and not force:
            print(f"[predict] CACHE HIT -> {p.name}")
            return json.loads(p.read_text(encoding="utf-8"))

        print(f"[predict] generating ({stage}, {split}, n={len(ds)})")
        feats = [adapter.preprocess(ex) for ex in ds]
        preds, refs = [], []
        adapter.model.eval()
        for i in range(0, len(feats), batch_size):
            b = adapter.collate(feats[i:i+batch_size])
            b_dev = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
            preds.extend(adapter.generate(b_dev)); refs.extend(b["text"])
            print(f"  {min(i+batch_size,len(feats))}/{len(feats)}", end="\r")
        rec = {"model": adapter.name, "split": split, "stage": stage,
               "predictions": preds, "references": refs,
               "n": len(preds), "ts": time.time()}
        p.write_text(json.dumps(rec, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"\n[predict] saved -> {p.name}")
        return rec

## Cell 10 — EvaluateAPI (cached)

In [ ]:
class EvaluateAPI:
    @staticmethod
    def _path(model_name, split, stage):
        return METRIC_DIR / f"{model_name.replace('/','__')}__{split}__{stage}.json"

    @staticmethod
    def run(model_name, pred_record, split="test", stage="base", force=False):
        p = EvaluateAPI._path(model_name, split, stage)
        if p.exists() and not force:
            m = json.loads(p.read_text()); print(f"[eval] CACHE HIT -> {m}"); return m
        m = compute_wer_cer(pred_record["predictions"], pred_record["references"])
        m.update({"model": model_name, "split": split, "stage": stage})
        p.write_text(json.dumps(m, indent=2))
        print(f"[eval] WER={m['wer']:.4f} CER={m['cer']:.4f} (n={m['n']}) -> {p.name}")
        return m

## Cell 11 — Build adapter + load base model

In [ ]:
set_seed()
adapter = get_adapter(MODEL_NAME, lang=LANG)
adapter.load_base()
n_params = sum(p.numel() for p in adapter.model.parameters())
print(f"{MODEL_NAME}: {n_params/1e6:.1f}M params | loss_type={adapter.loss_type} | unsloth={adapter.supports_unsloth}")

## Cell 12 — Baseline preds + eval on test (cached)

In [ ]:
base_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="base",
                              batch_size=ConfigAPI.train(MODEL_NAME).per_device_eval_batch_size)
base_metrics = EvaluateAPI.run(MODEL_NAME, base_preds, split="test", stage="base")

for p, r in list(zip(base_preds["predictions"], base_preds["references"]))[:3]:
    print(f"REF : {r}\nHYP : {p}\n")

## Cell 13 — Apply LoRA

In [ ]:
lora_spec  = ConfigAPI.lora(MODEL_NAME)
train_spec = ConfigAPI.train(MODEL_NAME)
if SMOKE_TEST:
    train_spec.num_epochs = 2
    train_spec.early_stopping_patience = 4

adapter.apply_lora(lora_spec)

## Cell 14 — TrainAPI

Custom loop rather than `Seq2SeqTrainer`: the fairseq2 module isn't a `PreTrainedModel`, so the HF Trainer's save/load/generate hooks don't apply. Per-epoch logging of train loss, val loss, val WER, val CER; early stopping on WER with patience 4; best-WER checkpoint only.

In [ ]:
import wandb
from contextlib import nullcontext
from torch.utils.data import DataLoader

def _amp(spec):
    """bf16 autocast on CUDA; no-op elsewhere so the loop also runs on CPU."""
    if DEVICE == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return nullcontext()

class _ListDS(torch.utils.data.Dataset):
    def __init__(self, feats): self.f = feats
    def __len__(self): return len(self.f)
    def __getitem__(self, i): return self.f[i]

class TrainAPI:
    @staticmethod
    def _prep(adapter, ds, spec):
        """Pre-flight: preprocess + drop over-long / empty items before the loop."""
        feats, dropped = [], 0
        for ex in ds:
            f = adapter.preprocess(ex)
            if f["audio_len"] > spec.max_audio_seconds: dropped += 1; continue
            if not f["text"].strip():                   dropped += 1; continue
            if len(f["labels"]) > spec.max_label_tokens:
                f["labels"] = f["labels"][:spec.max_label_tokens]
            feats.append(f)
        print(f"[prep] kept {len(feats)}, dropped {dropped}")
        return feats

    @staticmethod
    @torch.no_grad()
    def _validate(adapter, loader, spec):
        adapter.model.eval(); losses, preds, refs = [], [], []
        for b in loader:
            g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
            try:
                with _amp(spec):
                    losses.append(float(adapter.train_step(
                        {k: v for k, v in g.items()
                         if k in ("input_values","attention_mask","labels","label_lengths","lang")})))
            except Exception as e:
                print(f"[val] loss skipped: {e}")
            preds.extend(adapter.generate(g)); refs.extend(b["text"])
        m = compute_wer_cer(preds, refs)
        m["val_loss"] = float(np.mean(losses)) if losses else float("nan")
        return m

    @staticmethod
    def run(adapter, splits, spec: TrainConfigSpec, lora_spec: LoRAConfigSpec):
        slug = adapter.name.replace("/", "__")
        run  = wandb.init(project=os.environ["WANDB_PROJECT"], name=f"{slug}-lora",
                          config={**asdict(spec), **asdict(lora_spec),
                                  "model": adapter.name, "lang": LANG, "smoke": SMOKE_TEST},
                          reinit=True)
        best_dir = CKPT_DIR / slug / "best"; best_dir.mkdir(parents=True, exist_ok=True)

        tr_f = TrainAPI._prep(adapter, splits["train"], spec)
        va_f = TrainAPI._prep(adapter, splits["validation"], spec)
        tr = DataLoader(_ListDS(tr_f), batch_size=spec.per_device_train_batch_size, shuffle=True,
                        collate_fn=adapter.collate, num_workers=spec.dataloader_num_workers,
                        pin_memory=(DEVICE=="cuda"), drop_last=False)
        va = DataLoader(_ListDS(va_f), batch_size=spec.per_device_eval_batch_size, shuffle=False,
                        collate_fn=adapter.collate, num_workers=2)

        params = [p for p in adapter.model.parameters() if p.requires_grad]
        opt = None
        if DEVICE == "cuda":            # bitsandbytes 8-bit optimizers are CUDA-only
            try:
                import bitsandbytes as bnb
                opt = bnb.optim.AdamW8bit(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)
            except Exception as e:
                print(f"[opt] AdamW8bit unavailable ({e}); using torch.AdamW")
        if opt is None:
            opt = torch.optim.AdamW(params, lr=spec.learning_rate, weight_decay=spec.weight_decay)

        steps_pe = max(1, math.ceil(len(tr) / spec.gradient_accumulation_steps))
        total    = steps_pe * spec.num_epochs
        from transformers import get_linear_schedule_with_warmup
        sched = get_linear_schedule_with_warmup(opt, int(total*spec.warmup_ratio), total)

        best_wer, bad_epochs, gstep, history = float("inf"), 0, 0, []

        for epoch in range(1, spec.num_epochs + 1):
            adapter.model.train(); ep_loss, nb = 0.0, 0
            opt.zero_grad(set_to_none=True)
            for i, b in enumerate(tr):
                g = {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in b.items()}
                with _amp(spec):
                    loss = adapter.train_step(g) / spec.gradient_accumulation_steps
                if not torch.isfinite(loss):
                    print(f"[nan] step {i} non-finite loss, skipping batch")
                    opt.zero_grad(set_to_none=True); continue
                loss.backward()
                if (i + 1) % spec.gradient_accumulation_steps == 0 or (i + 1) == len(tr):
                    gnorm = torch.nn.utils.clip_grad_norm_(params, spec.max_grad_norm)
                    if not torch.isfinite(gnorm):
                        print(f"[nan] step {i} non-finite grad norm, skipping update")
                        opt.zero_grad(set_to_none=True); continue
                    opt.step(); sched.step(); opt.zero_grad(set_to_none=True); gstep += 1
                    wandb.log({"train/step_loss": float(loss)*spec.gradient_accumulation_steps,
                               "train/grad_norm": float(gnorm),
                               "train/lr": sched.get_last_lr()[0]}, step=gstep)
                ep_loss += float(loss) * spec.gradient_accumulation_steps; nb += 1

            train_loss = ep_loss / max(nb, 1)
            vm = TrainAPI._validate(adapter, va, spec)
            row = {"epoch": epoch, "train_loss": train_loss, "val_loss": vm["val_loss"],
                   "val_wer": vm["wer"], "val_cer": vm["cer"]}
            history.append(row)
            wandb.log({"epoch": epoch, "train/loss": train_loss, "val/loss": vm["val_loss"],
                       "val/wer": vm["wer"], "val/cer": vm["cer"]}, step=gstep)
            print(f"epoch {epoch:>3} | train {train_loss:.4f} | val {vm['val_loss']:.4f} "
                  f"| WER {vm['wer']:.4f} | CER {vm['cer']:.4f}")

            # ---- best-WER checkpoint + early stopping (patience 4) ----
            if vm["wer"] < best_wer - 1e-6:
                best_wer, bad_epochs = vm["wer"], 0
                try: adapter.model.save_pretrained(str(best_dir))
                except Exception:
                    torch.save({k: v for k, v in adapter.model.state_dict().items() if "lora" in k},
                               best_dir / "adapter.pt")
                (best_dir / "best.json").write_text(json.dumps({**row, "gstep": gstep}, indent=2))
                print(f"  -> new best WER {best_wer:.4f}, saved to {best_dir}")
            else:
                bad_epochs += 1
                print(f"  -> no improvement ({bad_epochs}/{spec.early_stopping_patience})")
                if bad_epochs >= spec.early_stopping_patience:
                    print(f"[early-stop] epoch {epoch}, best WER {best_wer:.4f}"); break

        wandb.summary["best_val_wer"] = best_wer
        (CKPT_DIR / slug / "history.json").write_text(json.dumps(history, indent=2))
        return {"best_wer": best_wer, "best_dir": str(best_dir), "history": history, "run": run}

## Cell 15 — Train

In [ ]:
set_seed()
train_out = TrainAPI.run(adapter, SPLITS, train_spec, lora_spec)
print(f"best val WER: {train_out['best_wer']:.4f} @ {train_out['best_dir']}")

## Cell 16 — Load best checkpoint, predict + evaluate on test, save

In [ ]:
best = Path(train_out["best_dir"])
try:
    from peft import PeftModel
    adapter.model.load_adapter(str(best), adapter_name="default")
    print(f"[ckpt] loaded best adapter <- {best}")
except Exception:
    sd = torch.load(best / "adapter.pt", map_location=DEVICE)
    adapter.model.load_state_dict(sd, strict=False)
    print(f"[ckpt] loaded best state_dict <- {best}")

tuned_preds   = PredictAPI.run(adapter, SPLITS["test"], split="test", stage="tuned",
                               batch_size=train_spec.per_device_eval_batch_size, force=True)
tuned_metrics = EvaluateAPI.run(MODEL_NAME, tuned_preds, split="test", stage="tuned", force=True)

summary = {
    "model": MODEL_NAME, "lang": LANG, "smoke_test": SMOKE_TEST,
    "base":  {"wer": base_metrics["wer"],  "cer": base_metrics["cer"]},
    "tuned": {"wer": tuned_metrics["wer"], "cer": tuned_metrics["cer"]},
    "delta": {"wer": base_metrics["wer"] - tuned_metrics["wer"],
              "cer": base_metrics["cer"] - tuned_metrics["cer"]},
    "best_val_wer": train_out["best_wer"],
    "lora": asdict(lora_spec), "train": asdict(train_spec),
}
sp = METRIC_DIR / f"{MODEL_NAME.replace('/','__')}__SUMMARY.json"
sp.write_text(json.dumps(summary, indent=2, ensure_ascii=False))

wandb.log({"test/base_wer": base_metrics["wer"],   "test/base_cer": base_metrics["cer"],
           "test/tuned_wer": tuned_metrics["wer"], "test/tuned_cer": tuned_metrics["cer"]})
wandb.log({"test/predictions": wandb.Table(
    columns=["reference", "base_hyp", "tuned_hyp"],
    data=[[r, b, t] for r, b, t in zip(tuned_preds["references"],
                                       base_preds["predictions"],
                                       tuned_preds["predictions"])])})
wandb.finish()
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Cell 17 — Smoke test both OmniASR models

Same code path, only `MODEL_NAME` changes. This is the one-shot you asked for: since 1B and 300M are architecturally identical, the loop below reuses `OmniASRAdapter` unchanged.

In [ ]:
def smoke(model_name, splits):
    set_seed()
    a = get_adapter(model_name, lang=LANG); a.load_base()
    bp = PredictAPI.run(a, splits["test"], "test", "base", batch_size=2)
    bm = EvaluateAPI.run(model_name, bp, "test", "base")
    ls, ts = ConfigAPI.lora(model_name), ConfigAPI.train(model_name)
    ts.num_epochs = 2; ts.per_device_train_batch_size = 1; ts.gradient_accumulation_steps = 2
    a.apply_lora(ls)
    out = TrainAPI.run(a, splits, ts, ls)
    tp = PredictAPI.run(a, splits["test"], "test", "tuned", batch_size=2, force=True)
    tm = EvaluateAPI.run(model_name, tp, "test", "tuned", force=True)
    del a.model, a; gc.collect(); torch.cuda.empty_cache()
    return {"model": model_name, "base_wer": bm["wer"], "tuned_wer": tm["wer"],
            "best_val_wer": out["best_wer"], "status": "PASS"}

results = []
for m in ["omnilingual-asr/omniASR_LLM_300M", "omnilingual-asr/omniASR_LLM_1B"]:
    try:
        results.append(smoke(m, SPLITS))
    except Exception as e:
        import traceback; traceback.print_exc()
        results.append({"model": m, "status": f"FAIL: {e}"})
    print("=" * 70)

import pandas as pd
pd.DataFrame(results)